# Ремонт STL до водонепроницаемости (ГМЗ Царское Село)

Чиним децимированный скан (2,07 млн треугольников) для 3D-печати:

1. скачиваем STL с tzar.ostrov-vezeniya.ru;
2. **пред-чистка PyMeshLab** — убирает дубли и не-манифолдные рёбра (без неё MeshFix жуёт часами — проверено);
3. **MeshFix** — закрывает дыры;
4. проверка герметичности (каждое ребро должно принадлежать ровно 2 граням);
5. скачиваем результат. Если MeshFix не справился — **план Б** внизу (Blender voxel remesh, водонепроницаемость гарантирована).

Запуск: Runtime → Run all. Обычный CPU-runtime, GPU не нужен.

In [ ]:
# 1. Зависимости (~1-2 мин)
!pip -q install pymeshfix pymeshlab numpy

In [ ]:
# 2. Входной файл
INPUT_URL = "https://tzar.ostrov-vezeniya.ru/v2/uploads/grot_decim.stl"  # 100 МБ, 2.07 млн трис
SRC, OUT = "input.stl", "repaired_watertight.stl"

import os, urllib.request
if not os.path.exists(SRC):
    print("скачиваю…")
    urllib.request.urlretrieve(INPUT_URL, SRC)
print(f"{os.path.getsize(SRC)/1e6:.1f} МБ")
# Вариант: свой файл — раскомментируй:
# from google.colab import files; up = files.upload(); SRC = next(iter(up))

In [ ]:
# 3. Чтение/запись/проверка бинарного STL (numpy, из нашего конвейера)
import numpy as np, struct
REC = np.dtype([("n", "<f4", 3), ("v", "<f4", (3, 3)), ("a", "<u2")])

def read_welded(path):
    n = struct.unpack("<I", open(path, "rb").read(84)[80:84])[0]
    buf = np.fromfile(path, dtype=REC, count=n, offset=84)
    V = buf["v"].reshape(-1, 3)
    u, inv = np.unique(np.ascontiguousarray(V).view([("", "V12")]).ravel(), return_inverse=True)
    P = u.view(np.float32).reshape(-1, 3).astype(np.float64)
    F = inv.reshape(-1, 3).astype(np.int64)
    return P, F[(F[:, 0] != F[:, 1]) & (F[:, 1] != F[:, 2]) & (F[:, 0] != F[:, 2])]

def write_stl(path, P, F, tag=b"watertight (colab)"):
    tri = P[F].astype(np.float32)
    nrm = np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0])
    L = np.linalg.norm(nrm, axis=1, keepdims=True)
    np.divide(nrm, L, out=nrm, where=L > 0)
    out = np.zeros(len(F), dtype=REC)
    out["n"], out["v"] = nrm, tri
    with open(path, "wb") as f:
        f.write(tag.ljust(80)); f.write(struct.pack("<I", len(F))); f.write(out.tobytes())

def check(P, F, label=""):
    E = np.concatenate([F[:, [0, 1]], F[:, [1, 2]], F[:, [2, 0]]]).astype(np.uint64)
    a, b = E[:, 0], E[:, 1]
    und = np.where(a < b, (a << np.uint64(32)) | b, (b << np.uint64(32)) | a)
    _, c = np.unique(und, return_counts=True)
    bd, nm = int((c == 1).sum()), int((c > 2).sum())
    ok = bd == 0 and nm == 0
    print(f"{label}: {len(F):,} трис | границы {bd:,} | не-манифолд {nm:,} → "
          + ("WATERTIGHT \u2714" if ok else "не герметичен \u2718"))
    return ok

In [ ]:
# 4. Пред-чистка PyMeshLab — КЛЮЧЕВОЙ шаг (после неё MeshFix сходится)
import pymeshlab, time
P, F = read_welded(SRC)
check(P, F, "вход")
t = time.time()
ms = pymeshlab.MeshSet()
ms.add_mesh(pymeshlab.Mesh(P, F))
for filt, kw in [
    ("meshing_remove_duplicate_faces", {}),
    ("meshing_remove_null_faces", {}),
    ("meshing_repair_non_manifold_edges", {}),   # method по умолчанию: Remove Faces
    ("meshing_repair_non_manifold_vertices", {}),
    ("meshing_remove_unreferenced_vertices", {}),
]:
    try:
        getattr(ms, filt)(**kw)
    except Exception as e:                      # имена фильтров меняются между версиями
        print(f"  (пропущен {filt}: {e})")
m = ms.current_mesh()
P1, F1 = m.vertex_matrix(), m.face_matrix().astype(np.int64)
print(f"чистка: {time.time()-t:.0f} с")
check(P1, F1, "после чистки")

In [ ]:
# 5. MeshFix (закрытие дыр). На чистой сетке — минуты.
import pymeshfix, time
t = time.time()
mf = pymeshfix.MeshFix(P1, F1.astype(np.int32))
mf.repair(verbose=True, joincomp=False, remove_smallest_components=False)
P2, F2 = mf.v, mf.f.astype(np.int64)
print(f"MeshFix: {time.time()-t:.0f} с, {len(F2):,} трис ({len(F2)/len(F1)*100:.1f}% граней сохранено)")
print("bbox до:", (P.max(0) - P.min(0)).round(0), " после:", (P2.max(0) - P2.min(0)).round(0))
ok = check(P2, F2, "итог")
write_stl(OUT, P2, F2)
print(f"записан {OUT} ({os.path.getsize(OUT)/1e6:.1f} МБ)")

In [ ]:
# 6. Забрать результат
from google.colab import files
files.download(OUT)
# Или сразу на сайт (ключ: ssh ostrov cat /srv/tzar-api.env):
# !curl -X PUT -H "X-Upload-Token: ВСТАВЬ_КЛЮЧ" --data-binary @repaired_watertight.stl \
#   "https://tzar.ostrov-vezeniya.ru/v2/api/upload?name=Грот · watertight&ext=stl"

---
## План Б: Blender voxel remesh (если MeshFix не дал ✔)

Строит **гарантированно водонепроницаемую** оболочку заново. Сетка становится равномерной:
детали сохраняются на уровне `VOXEL` мм (у нашего децимата шаг ~58 мм, так что 40 мм ничего не теряет).

In [ ]:
# !pip -q install bpy   # если не встал: попробуй bpy==4.2.0 (нужен python 3.11)
# import bpy, time
# VOXEL = 40.0  # мм
# t = time.time()
# bpy.ops.wm.read_factory_settings(use_empty=True)
# bpy.ops.wm.stl_import(filepath="input.stl")
# ob = bpy.context.selected_objects[0]
# bpy.context.view_layer.objects.active = ob
# ob.data.remesh_voxel_size = VOXEL
# bpy.ops.object.voxel_remesh()
# bpy.ops.wm.stl_export(filepath="voxel_watertight.stl", export_selected_objects=True)
# print(f"remesh: {time.time()-t:.0f} с")
# P3, F3 = read_welded("voxel_watertight.stl")
# check(P3, F3, "voxel remesh")
# from google.colab import files; files.download("voxel_watertight.stl")